# Análise do Desempenho Olímpico e Características Socioeconômicas

**Grupo:** Erick Souza Alberti, Felipe Augusto da Silva Cruz e Guilherme Henrique Pereira da Cunha

## Introdução

Os Jogos Olímpicos são o maior evento esportivo do mundo e possuem grande relevância esportiva, sociocultural e de projeção internacional para os países participantes. O desempenho de cada país costuma ser resumido pelo número absoluto de medalhas, uma medida que pode dificultar comparações entre nações com populações, economias e estruturas esportivas muito diferentes.

Neste trabalho, investigamos **possíveis relações entre o desempenho olímpico dos países e características socioeconômicas**, especialmente população total, Produto Interno Bruto (PIB) e PIB per capita. A análise se concentra nos **Jogos Olímpicos de Verão**.

Como as fontes possuem coberturas temporais diferentes, serão utilizados dois recortes. Nas perguntas que dependem apenas dos dados olímpicos, serão consideradas todas as edições de Verão disponíveis, de **1896 a 2024**. Já nas perguntas que envolvem indicadores do World Bank, serão consideradas as edições de Verão entre **1960 e 2024**, período em que há sobreposição entre os dados olímpicos e os indicadores socioeconômicos utilizados.

Ao longo do estudo, o desempenho será medido de formas complementares, como número total de medalhas, participação no total de medalhas de cada edição e medidas relativas, como medalhas por milhão de habitantes. As relações encontradas serão interpretadas como **associações**, e não como evidência de causalidade.


**Link do Repositório no GitHub:** https://github.com/felipeascruz/AEDV

## Descrição dos Dados

O trabalho utiliza três fontes: dois arquivos CSV com dados olímpicos e indicadores socioeconômicos obtidos pela API do World Bank.

* **Medal Table:** contém o resultado agregado de cada país em cada edição dos Jogos Olímpicos de Verão e de Inverno, de 1896 a 2026. Cada linha corresponde a um país em uma edição ou subedição, e aparecem apenas países que conquistaram ao menos uma medalha. A base possui 1.929 linhas e 11 colunas. Como as medalhas de esportes coletivos são contabilizadas uma única vez, esta tabela será utilizada como fonte principal para o total oficial de medalhas. Em 1956, a subedição equestre de Estocolmo aparece separadamente da edição de Melbourne; por isso, os registros serão agregados por **ano e NOC** antes das análises.

* **Olympic Athletes:** contém 315.094 registros de participação de atletas, com 16 colunas. Cada linha representa um atleta em um evento, de modo que um mesmo atleta pode aparecer diversas vezes. Esta base será usada principalmente para identificar **todos os países participantes em cada edição**, inclusive aqueles que conquistaram zero medalhas.

* **World Bank (API):** fornece indicadores anuais por país. Serão utilizados população total, PIB em dólares correntes e PIB per capita em dólares correntes. O arquivo coletado possui dados entre 1960 e 2025; como a última edição de Verão disponível é 2024, as análises socioeconômicas utilizarão as edições entre **1960 e 2024**.

### Combinação das bases

A unidade principal de análise será **país × edição olímpica**. Primeiro, a lista de participantes será obtida da base de atletas. Em seguida, o quadro de medalhas será agregado por `year` e `noc` e combinado à lista de participantes com uma junção à esquerda. Países participantes que não aparecem no quadro de medalhas receberão valor zero para ouro, prata, bronze e total, evitando que países sem medalhas sejam excluídos das análises.

Os códigos do World Bank seguem majoritariamente o padrão ISO, enquanto os dados olímpicos utilizam códigos NOC. Será feita uma padronização explícita dos identificadores antes da junção por `year + noc`. Casos históricos ou delegações sem correspondência socioeconômica defensável serão mantidos como dados ausentes, em vez de receberem uma correspondência arbitrária. Os valores ausentes de população, PIB e PIB per capita também **não serão removidos globalmente**: cada pergunta utilizará apenas as observações que possuírem os indicadores necessários para aquela análise.


In [ ]:
# Coleta dos indicadores socioeconômicos do World Bank
import wbgapi as wb

# Códigos oficiais das variáveis de interesse:
# SP.POP.TOTL: População total
# NY.GDP.MKTP.CD: PIB (US$ correntes)
# NY.GDP.PCAP.CD: PIB per capita (US$ correntes)
indicadores_wb = {
    'SP.POP.TOTL': 'population',
    'NY.GDP.MKTP.CD': 'gdp',
    'NY.GDP.PCAP.CD': 'gdp_per_capita'
}

# Os indicadores utilizados possuem cobertura no arquivo a partir de 1960.
# A coleta vai até 2025, embora as análises olímpicas terminem em 2024.
df_world_bank = wb.data.DataFrame(
    indicadores_wb.keys(),
    time=range(1960, 2026),
    labels=True,
    numericTimeKeys=True
).reset_index()

# Salva o CSV sem o índice numérico para evitar uma coluna extra na leitura posterior.
df_world_bank.to_csv('world_bank.csv', index=False)


In [ ]:
# Importação, limpeza e combinação das bases
import pandas as pd
from IPython.display import display

df_medalhas = pd.read_csv('medal_table.csv')
df_atletas = pd.read_csv('olympic_athletes.csv')
df_world_bank = pd.read_csv('world_bank.csv')

# -----------------------------
# 1. Preparação do World Bank
# -----------------------------

# Transforma as colunas de anos em uma única coluna 'year'.
df_world_bank = df_world_bank.melt(
    id_vars=['economy', 'series', 'Country', 'Series'],
    var_name='year',
    value_name='valor'
)

# Pivota a tabela para que população, PIB e PIB per capita tenham colunas próprias.
df_world_bank = df_world_bank.pivot_table(
    index=['economy', 'Country', 'year'],
    columns='series',
    values='valor'
).reset_index()

df_world_bank['year'] = df_world_bank['year'].astype(int)
df_world_bank = df_world_bank.rename(columns=indicadores_wb)
df_world_bank = df_world_bank.rename(columns={'economy': 'iso', 'Country': 'country'})

# O código CHI do World Bank corresponde a Channel Islands e conflita com o NOC CHI do Chile.
# Channel Islands não compete como um NOC próprio, então esse registro é removido antes da conversão.
df_world_bank = df_world_bank[df_world_bank['iso'] != 'CHI'].copy()

# Conversão dos códigos ISO do World Bank para os códigos NOC utilizados na base olímpica.
iso_para_noc = {
    'DZA': 'ALG', 'AGO': 'ANG', 'ATG': 'ANT', 'ABW': 'ARU', 'ASM': 'ASA', 'BHS': 'BAH', 'BGD': 'BAN', 'BRB': 'BAR',
    'BMU': 'BER', 'BTN': 'BHU', 'BLZ': 'BIZ', 'BWA': 'BOT', 'BHR': 'BRN', 'BRN': 'BRU', 'BGR': 'BUL', 'BFA': 'BUR',
    'KHM': 'CAM', 'CYM': 'CAY', 'COG': 'CGO', 'TCD': 'CHA', 'CHL': 'CHI', 'CRI': 'CRC', 'HRV': 'CRO', 'DNK': 'DEN',
    'SLV': 'ESA', 'FJI': 'FIJ', 'GMB': 'GAM', 'GNB': 'GBS', 'GNQ': 'GEQ', 'DEU': 'GER', 'GRC': 'GRE', 'GRD': 'GRN',
    'GTM': 'GUA', 'GIN': 'GUI', 'HTI': 'HAI', 'HND': 'HON', 'IDN': 'INA', 'IRN': 'IRI', 'VIR': 'ISV', 'VGB': 'IVB',
    'XKX': 'KOS', 'SAU': 'KSA', 'KWT': 'KUW', 'LVA': 'LAT', 'LBY': 'LBA', 'LSO': 'LES', 'MDG': 'MAD', 'MYS': 'MAS',
    'MWI': 'MAW', 'MNG': 'MGL', 'MCO': 'MON', 'MUS': 'MRI', 'MRT': 'MTN', 'MMR': 'MYA', 'NIC': 'NCA', 'NLD': 'NED',
    'NPL': 'NEP', 'NGA': 'NGR', 'NER': 'NIG', 'OMN': 'OMA', 'PRY': 'PAR', 'PHL': 'PHI', 'PSE': 'PLE', 'PRT': 'POR',
    'PRI': 'PUR', 'ZAF': 'RSA', 'WSM': 'SAM', 'SYC': 'SEY', 'KNA': 'SKN', 'SVN': 'SLO', 'SLB': 'SOL', 'LKA': 'SRI',
    'SDN': 'SUD', 'CHE': 'SUI', 'TZA': 'TAN', 'TON': 'TGA', 'TGO': 'TOG', 'ARE': 'UAE', 'URY': 'URU', 'VUT': 'VAN',
    'VNM': 'VIE', 'VCT': 'VIN', 'ZMB': 'ZAM', 'ZWE': 'ZIM'
}

df_world_bank['noc'] = df_world_bank['iso'].replace(iso_para_noc)

# O código olímpico do Líbano mudou na própria base: LIB até 2016 e LBN a partir de 2020.
df_world_bank.loc[(df_world_bank['iso'] == 'LBN') & (df_world_bank['year'] <= 2016), 'noc'] = 'LIB'
df_world_bank.loc[(df_world_bank['iso'] == 'LBN') & (df_world_bank['year'] >= 2020), 'noc'] = 'LBN'

# -----------------------------
# 2. Preparação das bases olímpicas
# -----------------------------

# Padronizações que possuem correspondência direta para o objetivo do trabalho.
# A Equipe Unificada (EUN), por exemplo, não é convertida para um país específico.
correcoes_olimpicas = {
    'ROC': 'RUS',  # Comitê Olímpico Russo -> Rússia
    'OAR': 'RUS'   # Atletas Olímpicos da Rússia -> Rússia (relevante apenas em edições de Inverno)
}

df_medalhas['noc'] = df_medalhas['noc'].replace(correcoes_olimpicas)
df_atletas['noc'] = df_atletas['noc'].replace(correcoes_olimpicas)

# O projeto analisa apenas os Jogos Olímpicos de Verão.
df_medalhas = df_medalhas[df_medalhas['season'] == 'Summer'].copy()
df_atletas = df_atletas[df_atletas['season'] == 'Summer'].copy()

# Em 1956, Melbourne e a subedição equestre de Estocolmo aparecem separadamente.
# Somamos os registros para obter uma única observação país × edição.
df_medalhas_pais_edicao = (
    df_medalhas
    .groupby(['year', 'noc'], as_index=False)
    .agg(
        country=('country', 'first'),
        gold=('gold', 'sum'),
        silver=('silver', 'sum'),
        bronze=('bronze', 'sum'),
        total=('total', 'sum')
    )
)

# A base de atletas define quem participou de cada edição, inclusive países sem medalhas.
df_participantes = df_atletas[['year', 'noc']].drop_duplicates()

# Left join: todos os participantes são mantidos.
df_olimpico = df_participantes.merge(
    df_medalhas_pais_edicao,
    on=['year', 'noc'],
    how='left',
    validate='one_to_one'
)

# Ausência no quadro de medalhas significa participação com zero medalhas, não dado ausente.
colunas_medalhas = ['gold', 'silver', 'bronze', 'total']
df_olimpico[colunas_medalhas] = df_olimpico[colunas_medalhas].fillna(0).astype(int)

# -----------------------------
# 3. Integração com o World Bank
# -----------------------------

df_pais_edicao = df_olimpico.merge(
    df_world_bank,
    on=['year', 'noc'],
    how='left',
    suffixes=('_olimpico', '_wb'),
    validate='one_to_one'
)

# Nome do país para visualizações: prioriza o nome olímpico, depois o World Bank e, por fim, o NOC.
df_pais_edicao['country'] = (
    df_pais_edicao['country_olimpico']
    .fillna(df_pais_edicao['country_wb'])
    .fillna(df_pais_edicao['noc'])
)

# Recortes que serão usados nas perguntas.
df_olimpico_completo = df_pais_edicao.copy()  # 1896–2024: perguntas apenas olímpicas

df_socioeconomico = df_pais_edicao[
    df_pais_edicao['year'].between(1960, 2024)
].copy()  # 1960–2024: perguntas que usam World Bank

# Não removemos NaNs socioeconômicos aqui. Cada pergunta filtra apenas o indicador de que precisa.

print(
    f"Período olímpico (Verão): {df_olimpico_completo['year'].min()}–{df_olimpico_completo['year'].max()} | "
    f"{len(df_olimpico_completo)} observações país × edição"
)
print(
    f"Período para análises socioeconômicas: {df_socioeconomico['year'].min()}–{df_socioeconomico['year'].max()} | "
    f"{len(df_socioeconomico)} observações país × edição"
)

# Auditoria simples da disponibilidade dos indicadores no recorte socioeconômico.
display(df_socioeconomico[['population', 'gdp', 'gdp_per_capita']].isna().sum().rename('valores_ausentes'))


### 1. Como o desempenho olímpico dos países, medido pelo número de medalhas conquistadas, varia entre as diferentes edições dos Jogos analisadas?

**Transformação dos Dados:**

*(Documente aqui as agregações, derivações e limpezas feitas especificamente para responder a esta pergunta. Por exemplo, agrupar medalhas por equipe, ou derivar medalhas/milhão de habitantes)*

In [ ]:
# A visualização começará com Brasil, China e Estados Unidos
paises_de_interesse = {'BRA', 'CHN', 'USA'}


**Análise e Resposta:**

*(Interprete o gráfico gerado acima e responda diretamente à pergunta, lembrando de discutir em termos de associação e não causalidade.)*

### 2. Como as medalhas estão distribuídas entre os países participantes? Elas se concentram em um grupo relativamente pequeno de países?

**Transformação dos Dados:**

*(Documente aqui as agregações, derivações e limpezas feitas especificamente para responder a esta pergunta. Por exemplo, agrupar medalhas por equipe, ou derivar medalhas/milhão de habitantes)*

In [ ]:
# Código para transformação de dados e geração do gráfico para a 2ª Pergunta
# ...
# plt.show()

**Análise e Resposta:**

*(Interprete o gráfico gerado acima e responda diretamente à pergunta, lembrando de discutir em termos de associação e não causalidade.)*

### 3. Países com populações maiores tendem a conquistar mais medalhas?

**Transformação dos Dados:**

*(Documente aqui as agregações, derivações e limpezas feitas especificamente para responder a esta pergunta. Por exemplo, agrupar medalhas por equipe, ou derivar medalhas/milhão de habitantes)*

In [ ]:
# Código para transformação de dados e geração do gráfico para a 3ª Pergunta
# ...
# plt.show()

**Análise e Resposta:**

*(Interprete o gráfico gerado acima e responda diretamente à pergunta, lembrando de discutir em termos de associação e não causalidade.)*

### 4. Existe associação entre o PIB de um país e seu número de medalhas conquistadas?

**Transformação dos Dados:**

*(Documente aqui as agregações, derivações e limpezas feitas especificamente para responder a esta pergunta. Por exemplo, agrupar medalhas por equipe, ou derivar medalhas/milhão de habitantes)*

In [ ]:
# Código para transformação de dados e geração do gráfico para a 4ª Pergunta
# ...
# plt.show()

**Análise e Resposta:**

*(Interprete o gráfico gerado acima e responda diretamente à pergunta, lembrando de discutir em termos de associação e não causalidade.)*

### 5. A relação observada muda quando utilizamos PIB per capita em vez de PIB total?

**Transformação dos Dados:**

*(Documente aqui as agregações, derivações e limpezas feitas especificamente para responder a esta pergunta. Por exemplo, agrupar medalhas por equipe, ou derivar medalhas/milhão de habitantes)*

In [ ]:
# Código para transformação de dados e geração do gráfico para a 5ª Pergunta
# ...
# plt.show()

**Análise e Resposta:**

*(Interprete o gráfico gerado acima e responda diretamente à pergunta, lembrando de discutir em termos de associação e não causalidade.)*

### 6. Como a comparação entre os países muda quando utilizamos medidas relativas, como medalhas por milhão de habitantes, em vez do número absoluto de medalhas?

**Transformação dos Dados:**

*(Documente aqui as agregações, derivações e limpezas feitas especificamente para responder a esta pergunta. Por exemplo, agrupar medalhas por equipe, ou derivar medalhas/milhão de habitantes)*

In [ ]:
# Código para transformação de dados e geração do gráfico para a 6ª Pergunta
# ...
# plt.show()

**Análise e Resposta:**

*(Interprete o gráfico gerado acima e responda diretamente à pergunta, lembrando de discutir em termos de associação e não causalidade.)*

### 7. Quais países apresentam desempenho olímpico muito acima ou abaixo do padrão observado para países com características socioeconômicas semelhantes?

**Transformação dos Dados:**

*(Documente aqui as agregações, derivações e limpezas feitas especificamente para responder a esta pergunta. Por exemplo, agrupar medalhas por equipe, ou derivar medalhas/milhão de habitantes)*

In [ ]:
# Código para transformação de dados e geração do gráfico para a 7ª Pergunta
# ...
# plt.show()

**Análise e Resposta:**

*(Interprete o gráfico gerado acima e responda diretamente à pergunta, lembrando de discutir em termos de associação e não causalidade.)*

### 8. Essas relações entre características socioeconômicas e desempenho olímpico permanecem semelhantes ao longo das diferentes edições dos Jogos?

**Transformação dos Dados:**

*(Documente aqui as agregações, derivações e limpezas feitas especificamente para responder a esta pergunta. Por exemplo, agrupar medalhas por equipe, ou derivar medalhas/milhão de habitantes)*

In [ ]:
# Código para transformação de dados e geração do gráfico para a 8ª Pergunta
# ...
# plt.show()

**Análise e Resposta:**

*(Interprete o gráfico gerado acima e responda diretamente à pergunta, lembrando de discutir em termos de associação e não causalidade.)*

## Conclusão

*(Escreva a síntese final do trabalho, as recomendações e as limitações encontradas na análise, como por exemplo, problemas de padronização ao unir a base esportiva à do Banco Mundial.)*